In [3]:
#!/usr/bin/env python3
"""
Gem / cube counter
==================
Reads gem inventory screenshots, detects which of the 6 slots hold a gem (and
its colour), OCRs the quantity printed under each gem, and sums everything up.

Pipeline (each stage is its own function so you can edit/replace it freely):

    expand_archives()      zips / 7z (even nested) -> folders of screenshots
    crop_images()          full screenshots -> cropped strips
    count_gems()           one image -> {"red small": 40, ...}
        apply_color_thresholds()   HSV colour masks             <-- colour tuning lives here
        detect_gems_by_grid()      which slot has which colour
        ocr_all_slots()            read the number under each gem
        label_gem_sizes()          small / big + inventory dict
    cumulative_gem_count() every image in a folder -> grand total
    add_extra_gems()       optionally type in extra gems by hand

Install:
    pip install easyocr opencv-python numpy Pillow
    pip install pyunpack patool        # only needed for .7z input (.zip needs nothing extra)

Usage (any mix of zips, 7z files and folders; zips inside zips are opened too):
    python cube_counter.py Screenshots.zip
    python cube_counter.py a.zip b.zip some_folder/
    python cube_counter.py cropped_output/ --no-crop # already-cropped images
    python cube_counter.py a.zip --debug             # save masks / OCR crops
    python cube_counter.py a.zip --no-extras
"""

import argparse
import os
import re
import shutil
import sys
import zipfile
from collections import Counter, defaultdict

import cv2
import numpy as np
from PIL import Image


# =============================================================================
# CONFIG  -- the stuff you're most likely to tweak
# =============================================================================

# Crop box applied to each full screenshot: (left, top, right, bottom)
CROP_BOX = (363, 335, 1557, 560)

# Where things go
EXTRACT_DIR = "extracted_images"
CROPPED_DIR = "cropped_output"
DEBUG_DIR = "debug_output"

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".bmp")
ARCHIVE_EXTENSIONS = (".zip", ".7z")

# Layout of the cropped strip
NUM_SLOTS = 6                 # slots are equal-width columns
MIN_SLOT_PCT = 5.0            # % of a slot's pixels that must match a colour to count as a gem

# OCR region: the bottom strip of each slot, with side margins trimmed
TEXT_ROI_TOP_FRAC = 0.80      # keep the bottom 20% of the strip
TEXT_ROI_TRIM_X_FRAC = 0.15   # trim 15% off each side to drop the capsule border
OCR_UPSCALE = 3
OCR_ALLOWLIST = "0123456789.kK"
USE_GPU = True                # EasyOCR falls back to CPU on its own if no CUDA

# Output ordering / emoji
LABEL_ORDER = [
    "red small", "red big",
    "blue small", "blue big",
    "yellow small", "yellow big",
    "green small", "green big",
]
EMOJI = {
    "red small": "🔴",    "red big": "🟥",
    "blue small": "🔵",   "blue big": "🟦",
    "green small": "🟢",  "green big": "🟩",
    "yellow small": "🟡", "yellow big": "🟨",
}


# =============================================================================
# COLOUR DETECTION  -- edit the bounds here
# =============================================================================

def get_tuned_gem_bounds():
    """
    OpenCV HSV bounds (H: 0-179, S: 0-255, V: 0-255).

    Each colour maps to a LIST of (lower, upper) ranges, so a colour can have
    more than one band. That matters for red, whose hue wraps around 0/179:
    a single range can only cover one side of the wrap.

    High minimum saturation isolates vibrant gems from pale UI backgrounds.
    """
    return {
        "blue":   [([95, 120, 80],  [125, 255, 255])],
        "green":  [([60, 120, 80],  [95, 255, 255])],
        "yellow": [([18, 120, 100], [38, 255, 255])],
        "red":    [([160, 60, 80],  [179, 255, 255])],
        # If red is being missed, it probably also needs the low-hue side of
        # the wrap. Uncomment this second band (and tune it):
        #   ([0, 60, 80], [10, 255, 255]),
    }


def apply_color_thresholds(img_bgr):
    """Returns {color_name: binary_mask} for the strip image."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    masks = {}

    for color_name, ranges in get_tuned_gem_bounds().items():
        mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
        for lower, upper in ranges:
            mask |= cv2.inRange(hsv, np.array(lower), np.array(upper))
        masks[color_name] = mask

    return masks


# =============================================================================
# SLOT DETECTION
# =============================================================================

def _slot_slice(mask, slot_index, num_slots=NUM_SLOTS):
    """Returns the vertical column of `mask` belonging to one slot."""
    slot_width = mask.shape[1] // num_slots
    return mask[:, slot_index * slot_width:(slot_index + 1) * slot_width]


def detect_gems_by_grid(masks, min_pct_threshold=MIN_SLOT_PCT, num_slots=NUM_SLOTS):
    """
    Checks each slot column against every colour mask.
    Returns {slot_index: {"color", "slot_index", "signal_pct"}} for active slots.
    If several colours clear the threshold in one slot, the strongest wins.
    """
    slots = {}

    for color_name, mask in masks.items():
        for i in range(num_slots):
            slot = _slot_slice(mask, i, num_slots)
            pct = cv2.countNonZero(slot) / slot.size * 100

            if pct >= min_pct_threshold and (i not in slots or pct > slots[i]["signal_pct"]):
                slots[i] = {
                    "color": color_name,
                    "slot_index": i,
                    "signal_pct": round(pct, 1),
                }

    return slots


def inspect_slot_percentages(masks, num_slots=NUM_SLOTS):
    """Debug helper: prints the % of matching pixels per slot for each colour.
    Handy when tuning the HSV bounds."""
    for color_name, mask in masks.items():
        pcts = [
            round(cv2.countNonZero(_slot_slice(mask, i, num_slots)) / _slot_slice(mask, i, num_slots).size * 100, 1)
            for i in range(num_slots)
        ]
        print(f"[{color_name.upper()}] Slot %: {pcts}")


# =============================================================================
# OCR  (quantity under each gem)
# =============================================================================

_reader = None


def get_reader():
    """Creates the EasyOCR reader once, on first use."""
    global _reader
    if _reader is None:
        import easyocr  # imported lazily so the rest of the script works without it
        _reader = easyocr.Reader(["en"], gpu=USE_GPU)
    return _reader


def parse_quantity(text):
    """Converts strings like '40', '325', '1K', '4.1K', '2.5M' into ints."""
    if not text:
        return 0

    cleaned = text.upper().strip().replace(" ", "")
    match = re.search(r"(\d+(?:\.\d+)?)\s*([KMB])?", cleaned)
    if not match:
        return 0

    val_str, suffix = match.groups()
    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}
    return int(round(float(val_str) * multipliers.get(suffix, 1)))


def preprocess_slot_text(img_bgr, slot_index, num_slots=NUM_SLOTS):
    """Crops the quantity text under one slot and prepares it for OCR."""
    h, w, _ = img_bgr.shape
    slot_width = w // num_slots

    roi = img_bgr[int(h * TEXT_ROI_TOP_FRAC):h,
                  slot_index * slot_width:(slot_index + 1) * slot_width]

    # Trim side margins to remove dark capsule borders
    pad_x = int(roi.shape[1] * TEXT_ROI_TRIM_X_FRAC)
    roi = roi[:, pad_x:roi.shape[1] - pad_x]

    roi = cv2.resize(roi, (0, 0), fx=OCR_UPSCALE, fy=OCR_UPSCALE, interpolation=cv2.INTER_CUBIC)

    # Grayscale + contrast stretch, then back to 3-channel as EasyOCR expects
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)


def ocr_slot_quantity(img_bgr, slot_index, num_slots=NUM_SLOTS):
    """Reads the quantity for one slot. Returns (count, raw_text, processed_img)."""
    processed = preprocess_slot_text(img_bgr, slot_index, num_slots)

    results = get_reader().readtext(
        processed,
        allowlist=OCR_ALLOWLIST,
        detail=0,
        text_threshold=0.5,
        low_text=0.3,
    )
    raw_text = "".join(results)
    return parse_quantity(raw_text), raw_text, processed


def ocr_all_slots(img_bgr, slots, debug_dir=None, tag=""):
    """Adds a 'count' to every detected slot dict (in place) and returns it."""
    for slot_index, info in slots.items():
        count, raw_text, processed = ocr_slot_quantity(img_bgr, slot_index)
        info["count"] = count

        if debug_dir:
            print(f"  [debug] {tag} slot {slot_index}: OCR '{raw_text}' -> {count}")
            save_debug_image(debug_dir, f"{tag}_slot{slot_index}_ocr.png", processed)

    return slots


# =============================================================================
# SMALL / BIG LABELLING
# =============================================================================

def label_gem_sizes(slots):
    """
    Labels each slot's gem as 'small' or 'big' and builds the inventory dict.

    Each colour occupies a (small, big) pair of slots, so with two slots of the
    same colour the left one is small and the right one big. Otherwise falls
    back to slot parity (even index = small, odd = big).
    """
    color_groups = defaultdict(list)
    for slot_idx, info in slots.items():
        color_groups[info["color"]].append(slot_idx)

    for color, idxs in color_groups.items():
        idxs.sort()
        if len(idxs) == 2:
            slots[idxs[0]]["size"] = "small"
            slots[idxs[1]]["size"] = "big"
        else:
            for s in idxs:
                slots[s]["size"] = "small" if s % 2 == 0 else "big"

        for s in idxs:
            slots[s]["label"] = f"{slots[s]['color']} {slots[s]['size']}"

    inventory = {info["label"]: info["count"] for info in slots.values()}
    return slots, inventory


# =============================================================================
# SINGLE-IMAGE PIPELINE
# =============================================================================

def save_debug_image(debug_dir, filename, img):
    os.makedirs(debug_dir, exist_ok=True)
    cv2.imwrite(os.path.join(debug_dir, filename), img)


def count_gems(image_path, crop_box=None, min_pct_threshold=MIN_SLOT_PCT, debug=False):
    """
    Full pipeline for one screenshot.

    crop_box : optional (left, top, right, bottom) applied before processing
    debug    : if True, prints OCR results and saves masks + OCR crops to DEBUG_DIR

    Returns (slots, inventory), e.g. inventory = {"blue small": 40, "blue big": 1000, ...}
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    if crop_box is not None:
        l, t, r, b = crop_box
        img = img[t:b, l:r]

    debug_dir = DEBUG_DIR if debug else None
    tag = os.path.splitext(os.path.basename(image_path))[0]

    # 1. HSV colour masks
    masks = apply_color_thresholds(img)
    if debug:
        for color_name, mask in masks.items():
            save_debug_image(debug_dir, f"{tag}_mask_{color_name}.png", mask)
        print(f"[debug] {tag}")
        inspect_slot_percentages(masks)

    # 2. Which slots contain a gem, and what colour
    slots = detect_gems_by_grid(masks, min_pct_threshold=min_pct_threshold)

    # 3. OCR the quantity under each detected gem
    slots = ocr_all_slots(img, slots, debug_dir=debug_dir, tag=tag)

    # 4. Label small/big and build the inventory dict
    return label_gem_sizes(slots)


# =============================================================================
# BATCH PIPELINE  (works on lists of image paths)
# =============================================================================

def find_images(folder):
    """Recursively lists every image under a folder."""
    found = []
    for root, _dirs, filenames in os.walk(folder):
        for filename in sorted(filenames):
            if filename.lower().endswith(IMAGE_EXTENSIONS):
                found.append(os.path.join(root, filename))
    return found


def count_gems_in_paths(paths, **kwargs):
    """Runs count_gems on each image path. Returns {path: inventory}."""
    per_image = {}
    for path in paths:
        try:
            _, inventory = count_gems(path, **kwargs)
            per_image[path] = inventory
        except Exception as e:
            print(f"Failed on {path}: {e}")
    return per_image


def cumulative_gem_count(paths, **kwargs):
    """Counts every image and sums the inventories. Returns (totals, per_image)."""
    per_image = count_gems_in_paths(paths, **kwargs)
    total = Counter()
    for inventory in per_image.values():
        total.update(inventory)
    return dict(total), per_image


# =============================================================================
# PREP: ARCHIVES + CROP
# =============================================================================

def is_archive(name):
    return name.lower().endswith(ARCHIVE_EXTENSIONS)


def reset_dir(path):
    """Deletes and recreates a working folder so stale files from earlier runs
    never get counted."""
    shutil.rmtree(path, ignore_errors=True)
    os.makedirs(path, exist_ok=True)


def extract_archive(archive_path, dest_dir):
    """Extracts one archive into dest_dir. .zip uses the standard library;
    anything else (.7z) goes through pyunpack/patool."""
    os.makedirs(dest_dir, exist_ok=True)
    if archive_path.lower().endswith(".zip"):
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
    else:
        from pyunpack import Archive  # lazy: only needed for 7z etc.
        Archive(archive_path).extractall(dest_dir)


def expand_archives(sources, extract_dir=EXTRACT_DIR):
    """
    Takes any mix of folders and archives and returns a list of folders that
    contain the actual files. Archives are extracted into extract_dir (one
    subfolder each), and archives found inside folders or inside other
    archives are extracted too.
    """
    folders, queue, seen = [], [], set()

    def queue_archives_in(folder):
        for root, _dirs, filenames in os.walk(folder):
            for filename in filenames:
                if is_archive(filename):
                    queue.append(os.path.join(root, filename))

    for src in sources:
        if os.path.isdir(src):
            folders.append(src)
            queue_archives_in(src)
        elif is_archive(src):
            queue.append(src)
        else:
            print(f"Skipping {src} (not a folder or a supported archive)")

    n = 0
    while queue:
        archive = queue.pop(0)
        key = os.path.abspath(archive)
        if key in seen:
            continue
        seen.add(key)

        n += 1
        stem = os.path.splitext(os.path.basename(archive))[0]
        dest = os.path.join(extract_dir, f"{n:03d}_{stem}")
        print(f"Extracting {archive}...")
        try:
            extract_archive(archive, dest)
        except Exception as e:
            print(f"Failed to extract {archive}: {e}")
            continue

        folders.append(dest)
        queue_archives_in(dest)  # nested archives

    return folders


def crop_images(folders, output_dir=CROPPED_DIR, crop_box=CROP_BOX):
    """
    Crops every image found under `folders` into output_dir (flat).
    Duplicate filenames get a _2, _3... suffix instead of overwriting.
    Returns the list of cropped file paths.
    """
    os.makedirs(output_dir, exist_ok=True)
    cropped_paths = []

    for folder in folders:
        for img_path in find_images(folder):
            filename = os.path.basename(img_path)
            stem, ext = os.path.splitext(filename)
            out_path, k = os.path.join(output_dir, filename), 1
            while os.path.exists(out_path):
                k += 1
                out_path = os.path.join(output_dir, f"{stem}_{k}{ext}")
            try:
                with Image.open(img_path) as img:
                    img.crop(crop_box).save(out_path)
                cropped_paths.append(out_path)
            except Exception as e:
                print(f"Failed to process {img_path}: {e}")

    print(f"Successfully cropped {len(cropped_paths)} images!")
    return cropped_paths


# =============================================================================
# REPORTING + MANUAL EXTRAS
# =============================================================================

def print_totals(totals, title="Cumulative totals:"):
    """Prints totals in red, blue, yellow, green order (small then big)."""
    print(title)
    extras = [label for label in totals if label not in LABEL_ORDER]
    for label in LABEL_ORDER + sorted(extras):
        if label in totals:
            print(f"  {EMOJI.get(label, '⚪')} {label}: {totals[label]:,}")


def apply_extra_gems(totals, extras):
    """
    Adds manually entered quantities to totals. `extras` is
    {label: "1.5k"} (strings or ints). Blank / unreadable entries are skipped.
    No input() here, so a UI can call it directly.
    """
    totals = dict(totals)
    for label, value in extras.items():
        qty = value if isinstance(value, int) else parse_quantity(str(value))
        if qty:
            totals[label] = totals.get(label, 0) + qty
    return totals


def add_extra_gems(totals):
    """Command-line prompt for an extra quantity per gem type. Enter skips one."""
    print("Enter extra gems to add (press Enter to skip)\n")
    extras = {}

    for label in LABEL_ORDER:
        while True:
            text = input(f"{EMOJI[label]} {label}: ").strip()
            if text == "":
                break
            qty = parse_quantity(text)
            if qty == 0:
                print("  ❌ Couldn't read that, try again (e.g. 40, 1K, 4.1K).")
                continue
            extras[label] = qty
            print(f"  ✅ Added {qty:,}")
            break

    return apply_extra_gems(totals, extras)


# =============================================================================
# MAIN
# =============================================================================

def run_pipeline(sources, crop=True, debug=False):
    """Everything except printing/prompting. Returns (totals, per_image).
    A UI can call this directly."""
    if crop:
        reset_dir(EXTRACT_DIR)
        reset_dir(CROPPED_DIR)
        folders = expand_archives(sources, EXTRACT_DIR)
        paths = crop_images(folders, CROPPED_DIR)
    else:
        reset_dir(EXTRACT_DIR)
        folders = expand_archives(sources, EXTRACT_DIR)
        paths = [p for f in folders for p in find_images(f)]

    if not paths:
        print("No images found.")
        return {}, {}
    return cumulative_gem_count(paths, debug=debug)


def main(argv=None):
    """CLI entry point. In a notebook, call main(["/content/Screenshots.zip"])."""
    parser = argparse.ArgumentParser(description="Count gems from inventory screenshots.")
    parser.add_argument("sources", nargs="*",
                        help="One or more .zip/.7z files and/or folders of images")
    parser.add_argument("--no-crop", action="store_true",
                        help="Images are already cropped; skip the crop step")
    parser.add_argument("--debug", action="store_true",
                        help=f"Print OCR details and save masks/OCR crops to {DEBUG_DIR}/")
    parser.add_argument("--no-extras", action="store_true",
                        help="Skip the manual 'add extra gems' prompt")
    in_notebook = "ipykernel" in sys.modules or "google.colab" in sys.modules
    if argv is None and in_notebook:
        # Colab/Jupyter inject their own args (e.g. `-f kernel.json`); ignore them all.
        args = parser.parse_args([])
    else:
        args = parser.parse_args(argv)

    if not args.sources:
        if in_notebook:
            print('In a notebook, pass your files explicitly, e.g.\n'
                  '    main(["/content/Screenshots.zip"])')
            return
        parser.error("give at least one .zip/.7z file or folder")

    totals, _per_image = run_pipeline(args.sources, crop=not args.no_crop, debug=args.debug)
    print()
    print_totals(totals)

    if not args.no_extras:
        print()
        totals = add_extra_gems(totals)
        print()
        print_totals(totals, "Updated cumulative totals:")


if __name__ == "__main__":
    main()

In a notebook, pass your files explicitly, e.g.
    main(["/content/Screenshots.zip"])


#UI

In [ ]:
!pip install -q gradio

import sys
import gradio as gr
cc = sys.modules["__main__"]

def count(files):
    if not files:
        return {}, []
    paths = [getattr(f, "name", f) for f in files]
    totals, _ = cc.run_pipeline(paths)
    return totals, to_rows(totals)

def with_extras(totals, *extras):
    merged = cc.apply_extra_gems(totals or {}, dict(zip(cc.LABEL_ORDER, extras)))
    return to_rows(merged)

def to_rows(totals):
    return [[f"{cc.EMOJI.get(l, '')} {l}", totals[l]] for l in cc.LABEL_ORDER if l in totals]

with gr.Blocks() as app:
    totals_state = gr.State({})
    files = gr.File(file_count="multiple", file_types=[".zip", ".7z"], label="Screenshot zips")
    count_btn = gr.Button("1. Count gems")
    extras = [gr.Textbox(label=f"{cc.EMOJI[l]} {l}", placeholder="extra, e.g. 1.5k") for l in cc.LABEL_ORDER]
    extras_btn = gr.Button("2. Add extras")
    table = gr.Dataframe(headers=["Gem", "Total"])

    count_btn.click(count, files, [totals_state, table])
    extras_btn.click(with_extras, [totals_state, *extras], table)

app.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ba4192a47212efbb3f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
app.close()

#Build/test

In [ ]:
!pip install easyocr
# Install required libraries for handling .7z files
!pip install Pillow pyunpack patool > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 28.5 MB/s eta 0:00:00


In [ ]:
%rm -r cropped_output/

rm: cannot remove 'cropped_output/': No such file or directory


In [ ]:
import os
from PIL import Image
from pyunpack import Archive
from google.colab import files
import cv2
import numpy as np
from google.colab.patches import cv2_imshow
# import pytesseract
from collections import defaultdict
import re
import easyocr

reader = easyocr.Reader(["en"], gpu=True)

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
archive_path = "/content/Screenshots.7z"
extract_dir = "extracted_images"
output_dir = "cropped_output"

os.makedirs(extract_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# 1. Extract the specified archive path
print(f"Extracting {archive_path}...")
Archive(archive_path).extractall(extract_dir)
363,335
1557,560


Extracting /content/Screenshots.7z...


(1557, 560)

In [ ]:
# Crop box coordinates: Left, Top, Right, Bottom
CROP_BOX = (363, 335, 1557, 560)

# 2. Find and crop all images recursively from the extracted folder
count = 0
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp")

for root, dirs, files_in_dir in os.walk(extract_dir):
    for filename in files_in_dir:
        if filename.lower().endswith(valid_extensions):
            img_path = os.path.join(root, filename)
            output_path = os.path.join(output_dir, filename)
            try:
                with Image.open(img_path) as img:
                    cropped_img = img.crop(CROP_BOX)
                    cropped_img.save(output_path)
                    count += 1
            except Exception as e:
                print(f"Failed to process {filename}: {e}")

print(f"Successfully cropped {count} images!")

Successfully cropped 54 images!


In [ ]:
def get_tuned_gem_bounds():
    """
    Hand-tuned OpenCV HSV bounds.
    High minimum saturation isolates vibrant gems from pale UI backgrounds.
    """
    return {
        "blue": {"lower": [95, 120, 80],"upper": [125, 255, 255]},
        "green": {"lower": [60, 120, 80],"upper": [95, 255, 255]},
        "yellow": {"lower": [18, 120, 100],"upper": [38, 255, 255]},
        "red": {"lower": [160, 60, 80],"upper": [179, 255, 255]}  # Standard Red lower band
    }

def apply_color_thresholds(cropped_row_img):
    hsv = cv2.cvtColor(cropped_row_img, cv2.COLOR_BGR2HSV)
    bounds = get_tuned_gem_bounds()
    masks = {}

    for color_name, limits in bounds.items():
        lower = np.array(limits["lower"])
        upper = np.array(limits["upper"])

        mask = cv2.inRange(hsv, lower, upper)
        masks[color_name] = mask

        # 🐛 DEBUG DISPLAY
        print(f"[{color_name.upper()} Mask] Lower: {lower}, Upper: {upper}")
        cv2_imshow(mask)

    return masks

In [ ]:
# img = cv2.imread('/content/cropped_output/Screenshot (180).png')
# apply_color_thresholds(img)


In [ ]:
def extract_gem_boxes(masks):
    """
    Cleans masks using a 3x3 kernel, extracts gem bounding boxes,
    and dynamically classifies gem sizes based on relative width comparisons.
    """
    kernel = np.ones((3, 3), np.uint8)
    detected_gems = []

    # 🧹 Step 1: Clean noise and collect bounding boxes across all masks
    for color_name, mask in masks.items():
        # Morphological opening removes speckles while keeping main shapes
        cleaned_mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        # 🐛 DEBUG DISPLAY: Display the cleaned mask for visual verification
        print(f"[{color_name.upper()} Cleaned Mask]")
        cv2_imshow(cleaned_mask)

        contours, _ = cv2.findContours(cleaned_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            # Filter out tiny residual artifacts (e.g., area < 50 pixels)
            if w * h > 50:
                detected_gems.append({
                    "color": color_name,
                    "bbox": (x, y, w, h),
                    "width": w,
                    "height": h
                })

    if not detected_gems:
        return []

    # 📊 Step 2: Relative size comparison using row statistics
    # Compute median width of all detected gems in this row
    median_width = np.median([g["width"] for g in detected_gems])

    for gem in detected_gems:
        # Gems significantly wider than the median are classified as "Big"
        gem["size"] = "Big" if gem["width"] > median_width * 1.2 else "Small"

    print(detected_gems)
    return detected_gems

In [ ]:
def inspect_slot_percentages(masks):
    """Divides masks into 6 horizontal slots and prints white pixel percentages."""
    for color_name, mask in masks.items():
        h, w = mask.shape
        slot_width = w // 6
        percentages = []

        for i in range(6):
            # Crop the current slot column
            slot = mask[:, i * slot_width : (i + 1) * slot_width]

            # Calculate percentage of white (255) pixels
            white_pixels = cv2.countNonZero(slot)
            total_pixels = slot.size
            pct = (white_pixels / total_pixels) * 100
            percentages.append(round(pct, 1))

        print(f"[{color_name.upper()}] Slot %: {percentages}")

# Run this on your thresholds output:
# inspect_slot_percentages(cleaned_masks)

In [ ]:
# masks=apply_color_thresholds(img)
# gem_count = extract_gem_boxes(masks)
# # inspect_slot_percentages(masks)

In [ ]:
def detect_gems_by_grid(masks, min_pct_threshold=5.0):
    """
    Checks each of the 6 slot columns across all color masks.
    Returns a list of active slots with detected gem color and size order.
    """
    slot_results = {}

    for color_name, mask in masks.items():
        h, w = mask.shape
        slot_width = w // 6

        for i in range(6):
            # Crop slot region
            slot = mask[:, i * slot_width : (i + 1) * slot_width]

            # Calculate percentage of white pixels
            white_pct = (cv2.countNonZero(slot) / slot.size) * 100

            if white_pct >= min_pct_threshold:
                # Slot 0,2,4 -> Small gem; Slot 1,3,5 -> Big gem (or based on order)
                slot_results[i] = {
                    "color": color_name,
                    "slot_index": i,
                    "signal_pct": round(white_pct, 1)
                }

    return slot_results

In [ ]:
# slots = detect_gems_by_grid(masks)

In [ ]:
# def ocr_slot_quantity(cropped_row_img, slot_input, total_slots=6):
#     """
#     Extracts quantities using Red-channel color contrast.
#     Handles integer slot indices, single slot dicts, or the full detected_slots dictionary.
#     """
#     # 🔁 If the full detected_slots dictionary was passed, process all active slots
#     if isinstance(slot_input, dict) and "slot_index" not in slot_input:
#         for s_idx, gem_info in slot_input.items():
#             count, _ = ocr_slot_quantity(cropped_row_img, gem_info, total_slots)
#             gem_info['count'] = count
#         return slot_input

#     # 🛠️ Extract integer index from a single slot dict or direct int
#     if isinstance(slot_input, dict):
#         slot_index = slot_input.get("slot_index", 0)
#     else:
#         slot_index = int(slot_input)

#     h, w, _ = cropped_row_img.shape
#     slot_width = w // total_slots

#     # ✂️ Slice bottom 20% height for slot_index
#     text_roi = cropped_row_img[int(h * 0.8) : h, slot_index * slot_width : (slot_index + 1) * slot_width]

#     # 🎯 1. Extract Red Channel (BGR index 2)
#     red_channel = text_roi[:, :, 2]

#     # 🔍 2. Upscale 4x with bicubic interpolation
#     # scaled = cv2.resize(red_channel, (0, 0), fx=4, fy=4, interpolation=cv2.INTER_CUBIC)

#     # 🔲 3. Trim outer 15% vertical and 10% horizontal edges to remove border frames
#     sh, sw = red_channel.shape
#     pad_y, pad_x = int(sh * 0.10), int(sw * 0.10)
#     trimmed = red_channel[pad_y : sh - pad_y, pad_x : sw - pad_x]

#     # 🔳 4. Add clean white border padding around the ROI
#     padded = cv2.copyMakeBorder(trimmed, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)


#     # 📝 5. Run OCR directly on the high-contrast grayscale image
#     custom_config = r'--psm 7 -c tessedit_char_whitelist=0123456789.kK'
#     digit_text = pytesseract.image_to_string(padded, config=custom_config).strip()

#     # 🖼️ Debug preview for every processed slot
#     print(f"\n--- [DEBUG] Slot Index {slot_index},count {digit_text} ---")
#     cv2_imshow(padded)

#     #
#     # return (int(digit_text) if digit_text.isdigit() else 0), padded
#     return (digit_text), padded

In [ ]:
# ocr_slot_quantity(img, slots)

In [ ]:
def parse_quantity(text: str) -> int:
    """Converts OCR strings like '1K', '4.1K', '325', '2.5M' into integer values."""
    if not text:
        return 0

    # Clean string and convert to uppercase
    cleaned = text.upper().strip().replace(" ", "")

    # Match numbers (including decimals) followed by an optional multiplier (K, M, B)
    match = re.search(r"(\d+(?:\.\d+)?)\s*([KMB])?", cleaned)
    if not match:
        return 0

    val_str, suffix = match.groups()
    val = float(val_str)

    multipliers = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}

    if suffix in multipliers:
        val *= multipliers[suffix]

    return int(round(val))

In [ ]:
# def ocr_slot_quantity(cropped_row_img, slot_input, total_slots=6):
#     """Extracts quantities using Red-channel color contrast and converts 'K/M/B' suffixes to integers."""
#     # 🔁 If full detected_slots dictionary was passed, process all active slots
#     if isinstance(slot_input, dict) and "slot_index" not in slot_input:
#         for s_idx, gem_info in slot_input.items():
#             count, _ = ocr_slot_quantity(cropped_row_img, gem_info, total_slots)
#             gem_info["count"] = count
#         return slot_input

#     # 🛠️ Extract integer index from single slot dict or direct int
#     if isinstance(slot_input, dict):
#         slot_index = slot_input.get("slot_index", 0)
#     else:
#         slot_index = int(slot_input)

#     h, w, _ = cropped_row_img.shape
#     slot_width = w // total_slots

#     # ✂️ Slice bottom 20% height for slot_index (Fixed missing '*' multiplication)
#     text_roi = cropped_row_img[
#         int(h * 0.8) : h, slot_index * slot_width : (slot_index + 1) * slot_width
#     ]

#     # 🎯 1. Extract Red Channel
#     red_channel = text_roi[:, :, 2]

#     # 🔍 2. Upscale 3x to give Tesseract clearer digit edges (fixes dropped digits like 35 -> 3)
#     scaled = cv2.resize(
#         red_channel, (0, 0), fx=3, fy=3, interpolation=cv2.INTER_CUBIC
#     )

#     # 🔲 3. Trim outer border frames
#     sh, sw = scaled.shape
#     pad_y, pad_x = int(sh * 0.10), int(sw * 0.10)
#     trimmed = scaled[pad_y : sh - pad_y, pad_x : sw - pad_x]

#     # 🔳 4. Add white padding around ROI
#     padded = cv2.copyMakeBorder(
#         trimmed, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255
#     )

#     # 📝 5. Run OCR with PSM 6 (single uniform block of text)
#     custom_config = r"--psm 6 -c tessedit_char_whitelist=0123456789.K"
#     raw_text = pytesseract.image_to_string(padded, config=custom_config).strip()

#     # 🔢 6. Parse string to integer (e.g., '1K' -> 1000, '4.1K' -> 4100)
#     parsed_count = parse_quantity(raw_text)

#     # 🖼️ Debug preview
#     print(
#         f"--- [DEBUG] Slot Index {slot_index} | Raw OCR: '{raw_text}' -> Parsed: {parsed_count} ---"
#     )
#     cv2_imshow(padded)

#     return parsed_count, padded

In [ ]:
def label_gem_sizes(detected_slots):
  """Labels slot gem sizes as 'small' or 'big' and builds single-image counts."""
  color_groups = defaultdict(list)
  for slot_idx, info in detected_slots.items():
    color_groups[info['color']].append(slot_idx)

  for color, slots in color_groups.items():
    slots.sort()
    if len(slots) == 2:
      detected_slots[slots[0]]['size'] = 'small'
      detected_slots[slots[1]]['size'] = 'big'
    elif len(slots) == 1:
      detected_slots[slots[0]]['size'] = (
          'small' if slots[0] % 2 == 0 else 'big'
      )
    else:
      # Fallback when > 2 slots match the same color
      for s in slots:
        detected_slots[s]['size'] = 'small' if s % 2 == 0 else 'big'

    for s in slots:
      data = detected_slots[s]
      data['label'] = f"{data['color']} {data['size']}"

  inventory = {
      info['label']: info['count'] for info in detected_slots.values()
  }
  return detected_slots, inventory

In [ ]:
def parse_quantity(text: str) -> int:
    """Converts strings like '40', '325', '1K', or '4.1K' into integers."""
    if not text:
        return 0

    cleaned = text.upper().strip().replace(" ", "")

    # Match numbers (including decimals) with optional 'K' suffix
    match = re.search(r"(\d+(?:\.\d+)?)\s*(K)?", cleaned)
    if not match:
        return 0

    val_str, suffix = match.groups()
    val = float(val_str)

    if suffix == "K":
        val *= 1_000

    return int(round(val))


def ocr_slot_quantity_easyocr(cropped_row_img, slot_input, total_slots=6):
    """Extracts slot quantities with EasyOCR using thresholding to stop 1K from reading as 4K."""
    if isinstance(slot_input, dict) and "slot_index" not in slot_input:
        for s_idx, gem_info in slot_input.items():
            count, _ = ocr_slot_quantity_easyocr(
                cropped_row_img, gem_info, total_slots
            )
            gem_info["count"] = count
        return slot_input

    if isinstance(slot_input, dict):
        slot_index = slot_input.get("slot_index", 0)
    else:
        slot_index = int(slot_input)

    h, w, _ = cropped_row_img.shape
    slot_width = w // total_slots

    # ✂️ Slice bottom 20% height for slot_index
    text_roi = cropped_row_img[
        int(h * 0.8) : h, slot_index * slot_width : (slot_index + 1) * slot_width
    ]

    # 🔲 Trim 15% horizontal margins to remove dark capsule borders
    rh, rw, _ = text_roi.shape
    pad_x = int(rw * 0.15)
    trimmed_roi = text_roi[:, pad_x : rw - pad_x]

    # 🔍 Upscale 3x
    scaled = cv2.resize(
        trimmed_roi, (0, 0), fx=3, fy=3, interpolation=cv2.INTER_CUBIC
    )

    # 🔳 Convert to Grayscale & Apply Adaptive Binarization
    gray = cv2.cvtColor(scaled, cv2.COLOR_BGR2GRAY)

    # Contrast Stretching (normalizes gray hues into sharp black/white boundaries)
    gray_norm = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)

    # Convert back to 3-channel RGB as expected by EasyOCR
    processed_img = cv2.cvtColor(gray_norm, cv2.COLOR_GRAY2RGB)

    # 📝 Run EasyOCR
    results = reader.readtext(
        processed_img,
        allowlist="0123456789.kK",
        detail=0,
        text_threshold=0.5,
        low_text=0.3,
    )

    raw_text = "".join(results)

    parsed_count = parse_quantity(raw_text)

    print(
        f"--- [DEBUG] Slot Index {slot_index} | Raw OCR: '{raw_text}' -> Parsed: {parsed_count} ---"
    )
    cv2_imshow(processed_img)

    return parsed_count, processed_img

In [ ]:
# gem_no_label_count = ocr_slot_quantity_easyocr(img, slots)
# label_gem_sizes(gem_no_label_count)

In [ ]:
import contextlib, io

def count_gems(image_path, crop_box=None, min_pct_threshold=5.0, debug=False):
    """
    Full pipeline for one screenshot.
    image_path : path to an image (already cropped, or pass crop_box to crop here)
    crop_box   : optional (left, top, right, bottom) crop applied before processing
    debug      : if False, silences the mask/OCR printouts and cv2_imshow previews
    Returns (detected_slots, inventory) e.g. inventory = {"blue small": 40, "blue big": 1000, ...}
    """
    global cv2_imshow
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    if crop_box is not None:
        l, t, r, b = crop_box
        img = img[t:b, l:r]

    original_imshow = cv2_imshow
    if not debug:
        cv2_imshow = lambda *a, **k: None  # silence previews in the helper functions
    try:
        with (contextlib.nullcontext() if debug else contextlib.redirect_stdout(io.StringIO())):
            # 1. HSV colour masks
            masks = apply_color_thresholds(img)
            # 2. Which of the 6 slots contain a gem, and what colour
            slots = detect_gems_by_grid(masks, min_pct_threshold=min_pct_threshold)
            # 3. OCR the quantity under each detected gem
            slots = ocr_slot_quantity_easyocr(img, slots)
            # 4. Label small/big and build the inventory dict
            slots, inventory = label_gem_sizes(slots)
    finally:
        cv2_imshow = original_imshow

    return slots, inventory


def count_gems_in_folder(folder=output_dir, **kwargs):
    """Runs count_gems on every image in a folder. Returns {filename: inventory}."""
    results = {}
    for name in sorted(os.listdir(folder)):
        if name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            try:
                _, inventory = count_gems(os.path.join(folder, name), **kwargs)
                results[name] = inventory
            except Exception as e:
                print(f"Failed on {name}: {e}")
    return results


In [ ]:
all_results = count_gems_in_folder('/content/cropped_output')
print(all_results)

{'Screenshot (180).png': {'green small': 20, 'green big': 0, 'red small': 48000, 'red big': 20}, 'Screenshot (181).png': {'green big': 440, 'green small': 400, 'red small': 48000, 'red big': 29}, 'Screenshot (182).png': {'blue small': 4, 'blue big': 400, 'green big': 407, 'green small': 100, 'yellow small': 2000, 'yellow big': 40}, 'Screenshot (183).png': {'blue small': 0, 'blue big': 9, 'green small': 0, 'green big': 0}, 'Screenshot (184).png': {'blue small': 0, 'blue big': 9, 'green big': 0, 'green small': 40, 'yellow small': 40}, 'Screenshot (185).png': {'blue small': 4, 'blue big': 9, 'green small': 40, 'green big': 0}, 'Screenshot (186).png': {'blue small': 0, 'blue big': 400, 'green big': 2, 'green small': 400}, 'Screenshot (187).png': {'green small': 0, 'green big': 0, 'yellow small': 3, 'yellow big': 9}, 'Screenshot (188).png': {'green small': 0, 'green big': 0, 'yellow small': 0, 'yellow big': 9}, 'Screenshot (189).png': {'blue small': 40, 'blue big': 9, 'green big': 4, 'green

In [ ]:
from collections import Counter

def cumulative_gem_count(folder=output_dir, **kwargs):
    """Runs count_gems on every image in the folder and sums the inventories."""
    total = Counter()
    per_image = {}

    for name in sorted(os.listdir(folder)):
        if not name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            continue
        try:
            _, inventory = count_gems(os.path.join(folder, name), **kwargs)
        except Exception as e:
            print(f"Failed on {name}: {e}")
            continue

        per_image[name] = inventory
        total.update(inventory)  # adds counts per label, e.g. "blue small"

    return dict(total), per_image


totals, per_image = cumulative_gem_count()

print("Cumulative totals:")
for label, count in sorted(totals.items()):
    print(f"  {label}: {count:,}")

Cumulative totals:
  blue big: 845
  blue small: 267
  green big: 1,497
  green small: 14,850
  red big: 78
  red small: 144,229
  yellow big: 561
  yellow small: 52,528


In [ ]:

EMOJI = {
    "red small": "🔴",    "red big": "🟥",
    "blue small": "🔵",   "blue big": "🟦",
    "green small": "🟢",  "green big": "🟩",
    "yellow small": "🟡", "yellow big": "🟨",
}

def cumulative_gem_count(folder=output_dir, **kwargs):
    """Runs count_gems on every image in the folder and sums the inventories."""
    total = Counter()
    per_image = {}

    for name in sorted(os.listdir(folder)):
        if not name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp")):
            continue
        try:
            _, inventory = count_gems(os.path.join(folder, name), **kwargs)
        except Exception as e:
            print(f"Failed on {name}: {e}")
            continue

        per_image[name] = inventory
        total.update(inventory)

    return dict(total), per_image


# totals, per_image = cumulative_gem_count()

print("Cumulative totals:")
for label, count in sorted(totals.items()):
    print(f"  {EMOJI.get(label, '⚪')} {label}: {count:,}")

Cumulative totals:
  🟦 blue big: 845
  🔵 blue small: 267
  🟩 green big: 1,497
  🟢 green small: 14,850
  🟥 red big: 78
  🔴 red small: 144,229
  🟨 yellow big: 561
  🟡 yellow small: 52,528


In [ ]:
LABEL_ORDER = [
    "red small", "red big",
    "blue small", "blue big",
    "yellow small", "yellow big",
    "green small", "green big",
]

def print_totals(totals, title="Cumulative totals:"):
    """Prints totals in red, blue, yellow, green order (small then big)."""
    print(title)
    # Known labels in the fixed order, then any unexpected labels at the end
    extras = [l for l in totals if l not in LABEL_ORDER]
    for label in LABEL_ORDER + sorted(extras):
        if label in totals:
            print(f"  {EMOJI.get(label, '⚪')} {label}: {totals[label]:,}")


def add_extra_gems(totals):
    """Prompts for an extra quantity for each gem type. Press Enter to skip one."""
    print("Enter extra gems to add (press Enter to skip)\n")

    for label in LABEL_ORDER:
        while True:
            text = input(f"{EMOJI[label]} {label}: ").strip()
            if text == "":
                break
            qty = parse_quantity(text)
            if qty == 0:
                print("  ❌ Couldn't read that, try again (e.g. 40, 1K, 4.1K).")
                continue
            totals[label] = totals.get(label, 0) + qty
            print(f"  ✅ Added {qty:,}")
            break

    return totals


totals_new = add_extra_gems(totals)
print()
print_totals(totals_new, "Updated cumulative totals:")

Enter extra gems to add (press Enter to skip)

🔴 red small: 2.2k
  ✅ Added 2,200
🟥 red big: 98
  ✅ Added 98
🔵 blue small: 1.5k
  ✅ Added 1,500
🟦 blue big: 238
  ✅ Added 238
🟡 yellow small: 245
  ✅ Added 245
🟨 yellow big: 753
  ✅ Added 753
🟢 green small: 508
  ✅ Added 508
🟩 green big: 164
  ✅ Added 164

Updated cumulative totals:
  🔴 red small: 146,429
  🟥 red big: 176
  🔵 blue small: 1,767
  🟦 blue big: 1,083
  🟡 yellow small: 52,773
  🟨 yellow big: 1,314
  🟢 green small: 15,358
  🟩 green big: 1,661
